In the following set of experiments, we aim to generate meaningful benchmarks for the dose escalation methods we would like to study.

### Utilities

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
from doseescalation.dose_escalator import (
    CRMDoseEscalator, 
    DoseEscalatorBase,
    ThreePlusThreeDoseEscalator, 
    UCBDoseEscalator,
    SEEDADoseEscalator,
    SEEDAPlateauDoseEscalator
)
from doseescalation.estimator import (
    AveragingEstimator
)
from doseescalation.evaluate import (
    plot_dose_proposals, 
    plot_acc_progression,
    plot_n_dles,
    simulate
)
from doseescalation.simulated_env import SimulatedEnv
from typing import Callable, Sequence

In [2]:
# Get the current timestamp for saving results:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [3]:
def a_key(a):
    return f"a = {a:.1f}"

def cohort_key(cohort):
    return f"Cohort {cohort + 1}"

In [4]:
def dose_toxic_curve(dose_levels, a_hat):
    return np.power((np.tanh(dose_levels) + 1) / 2, a_hat)

def inv_dose_toxic(p_dle, a):
    return np.arctanh(2 * np.power(p_dle, 1 / a) - 1)

In [5]:
N_LEVELS = 6
P_DLE_LEVELS = {
    0.4: [0.3, 0.4, 0.53, 0.62, 0.76, 0.87],
    1.0: [0.05, 0.1, 0.2, 0.3, 0.5, 0.7],
    1.3: [0.02, 0.05, 0.12, 0.3, 0.41, 0.63],
    3.4: [0.01, 0.02, 0.04, 0.08, 0.16, 0.3]
}
DOSE_LEVELS = {
    a: [inv_dose_toxic(v, a) for v in vs] 
    for a, vs in P_DLE_LEVELS.items()
}
N_A = len(DOSE_LEVELS.keys())
TTL = 0.32
N_TRIALS = 300

# To add more algorithms, extend the following list to include them.
ALGOS = ["3 + 3", "CRM", "UCB", "SEEDA", "SEEDA Plateau"]

COHORT_SIZE = 3
CORRECT_MTDS = {
    a_key(0.4): 0, 
    a_key(1.0): 3, 
    a_key(1.3): 3, 
    a_key(3.4): 5
}

# UCB parameter:
UCB_COEFF = 0.1

# SEEDA and SEEDA Plateau parameters:
P_HAT = (0.05, 0.15, 0.3, 0.35, 0.4, 0.45)
Q_HAT = np.array([0.33] * 6)
ETA = 2
# Number of patients who showed positive response must be <= COHORT_SIZE (≈ 33% efficacy rate):
N_EFFICATE = 1

# Dose-efficacy curve is the per-patient response ~ Bernoulli(q_dose),
# rising then plateauing (shared across all toxicity scenarios):
EFFICACY_PROBS = [0.1, 0.35, 0.6, 0.6, 0.6, 0.6]

# Efficacy env: reuse SimulatedEnv with "dose levels" = indices so the curve maps
# index -> probability, giving n_efficate ~ Binomial(cohort, EFFICACY_PROBS[idx]).
EFFICACY_ENV = SimulatedEnv(list(range(N_LEVELS)), lambda i: EFFICACY_PROBS[int(i)])

# Algorithms that consume efficacy: score these against the efficacy-optimal
# dose (plateau onset), not the toxicity MTD.
EFFICACY_ALGOS = ("SEEDA", "SEEDA Plateau")

def _optimal_dose(tox_probs, eff_probs, ttl):
    # k* = lowest safe dose achieving the max efficacy among the safe doses:
    safe = [k for k in range(len(tox_probs)) if tox_probs[k] <= ttl]
    if not safe:
        return 0
    best = max(eff_probs[k] for k in safe)
    return min(k for k in safe if eff_probs[k] == best)

OPTIMAL_DOSES = {
    a_key(a): _optimal_dose(P_DLE_LEVELS[a], EFFICACY_PROBS, TTL)
    for a in P_DLE_LEVELS
}

# Per-(scenario, algorithm) correct dose, for the plots: efficacy-aware methods
# are judged against the efficacy-optimal dose, the rest against the toxicity MTD.
CORRECT_DOSES = {
    a_key(a): {
        algo: (OPTIMAL_DOSES[a_key(a)] if algo in EFFICACY_ALGOS
               else CORRECT_MTDS[a_key(a)])
        for algo in ALGOS
    } for a in DOSE_LEVELS
}

In [6]:
def run_simulations(
    dose_escalator: DoseEscalatorBase,
    dose_levels: Sequence[float],
    dose_toxic_curve: Callable,
    cohort_size: int, 
    n_cohorts: int,
    n_efficate: int = 0,
    efficacy_env=None,
):
    env = SimulatedEnv(dose_levels, dose_toxic_curve)
    return simulate(
        cohort_sizes=[cohort_size] * n_cohorts, 
        dose_escalator=dose_escalator, 
        env=env,
        n_efficate=n_efficate,
        efficacy_env=efficacy_env
    )

In [7]:
def build_results_table(rec_map, alloc_map, correct_mtds, n_levels, algos,
                        opt_doses=None, efficacy_algos=()):
    records = []
    for scenario in rec_map:
        for algo in algos:
            # Efficacy-aware methods are scored against the efficacy-optimal dose:
            if opt_doses is not None and algo in efficacy_algos:
                correct = opt_doses[scenario]
            # Toxicity-only methods are scored against the toxicity MTD:
            else:
                correct = correct_mtds[scenario]
            recs = np.asarray(rec_map[scenario][algo])
            allocs = np.asarray(alloc_map[scenario][algo])
            for dose in range(n_levels):
                records.append({
                    "Scenario": scenario,
                    "Algorithm": algo,
                    "Dose": dose,
                    # Whether this dose is the algorithm's correct (target) dose:
                    "Is correct": dose == correct,
                    "Rec (in %)": round(100 * np.mean(recs == dose), 2) if recs.size else np.nan,
                    "Alloc (in %)": round(100 * np.mean(allocs == dose), 2) if allocs.size else np.nan
                })
    return pd.DataFrame.from_records(records)

### Simulations

We run 2 main sets of simulations, differing in the number of cohorts used in their experiments. 

The more realistic setting uses around 10^1 cohorts while the setting for asymptotic behaviours uses more than 10^2.

#### Realistic

In [8]:
N_REAL_COHORTS = 10

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [9]:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_REAL_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

def add_simulations(dose_escalator, a, algo, n_efficate=0, n_cohorts=N_REAL_COHORTS):
    allocations, recommendations, n_dles = run_simulations(
        dose_escalator,
        dose_levels,
        lambda dose: dose_toxic_curve(dose, a),
        COHORT_SIZE,
        n_cohorts=n_cohorts,
        n_efficate=n_efficate,
        efficacy_env=EFFICACY_ENV
    )
    # Determine the final declared MTD for this trial:
    a_algo_rec_map[a_key(a)][algo].append(recommendations[-1])

    # Determine every cohort's allocated dose pooled across trials:
    a_algo_alloc_map[a_key(a)][algo].extend(allocations)

    # Determine the total toxicities during this trial:
    a_algo_n_dle_map[a_key(a)][algo].append(sum(n_dles))

    # Determine the recommendation at each cohort:
    for cohort, rec in enumerate(recommendations):
        cohort_algo_rec_map[a_key(a)][cohort_key(cohort)][algo].append(rec)

# Run the realistic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0])
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1])
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2])
        
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE)

        seedapl_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_dose_escalator, a, ALGOS[4], N_EFFICATE)

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [10]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Realistic/Recommendations/real_algo_dose_rec_proposals.png"
)

Plot the proposals made for each cohort, so that we can see the time evolution of our `DoseEscalator`'s proposals. This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [11]:
for a in DOSE_LEVELS.keys():
    correct_mtds = {
        cohort_key(cohort): CORRECT_DOSES[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    mtds = {
        cohort_key(cohort): CORRECT_MTDS[a_key(a)]
        for cohort in range(N_REAL_COHORTS)
    }
    plot_dose_proposals(
        N_LEVELS, 
        N_TRIALS, 
        cohort_algo_rec_map[a_key(a)], 
        correct_mtds, 
        mtds=mtds,
        unit_width=100,
        title_text="MTD recommendations at each cohort",
        show_fig=False,
        img_path=f"plots/{timestamp}/Realistic/Recommendations/real_{a_key(a)}_dose_rec_proposal_progression.png"
    )

Plot the distribution of dose limiting events across all the trial runs and cohorts.

In [12]:
plot_n_dles(
    a_algo_n_dle_map, 
    img_path=f"plots/{timestamp}/Realistic/real_algo_n_dles.png"
)

Plot the distribution of dose allocations (the dose each cohort actually received), pooled across all cohorts and trial runs.

In [ ]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_REAL_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Realistic/Allocations/real_algo_dose_allocations.png"
)

Make a table with the metrics.

In [14]:
real_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, CORRECT_MTDS, N_LEVELS, ALGOS,
    opt_doses=OPTIMAL_DOSES, efficacy_algos=EFFICACY_ALGOS
)

In [15]:
real_table

,Scenario,Algorithm,Dose,Is correct,Rec (in %),Alloc (in %)
0,a = 0.4,3 + 3,0,True,85.00,82.23
1,a = 0.4,3 + 3,1,False,11.67,13.97
2,a = 0.4,3 + 3,2,False,2.33,2.90
3,a = 0.4,3 + 3,3,False,1.00,0.80
4,a = 0.4,3 + 3,4,False,0.00,0.10
...,...,...,...,...,...,...
115,a = 3.4,SEEDA Plateau,1,False,11.00,21.27
116,a = 3.4,SEEDA Plateau,2,True,0.33,33.50
117,a = 3.4,SEEDA Plateau,3,False,0.00,23.90
118,a = 3.4,SEEDA Plateau,4,False,0.00,7.53


In [16]:
# Paper-style wide layout (one column per dose level):
real_rec_pivot = real_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Rec (in %)", sort=False
)
real_alloc_pivot = real_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Alloc (in %)", sort=False
)

# Correct-dose summary: for each scenario/algorithm, the % recommending and
# allocating that algorithm's correct dose (efficacy-optimal for SEEDA/Plateau,
# toxicity MTD otherwise), plus whether that correct dose is the toxicity MTD.
correct_only = real_table[real_table["Is correct"]].copy()
correct_only["Is MTD"] = [
    dose == CORRECT_MTDS[scenario]
    for scenario, dose in zip(correct_only["Scenario"], correct_only["Dose"])
]
correct_summary = correct_only.set_index(["Scenario", "Algorithm"])[
    ["Dose", "Is MTD", "Rec (in %)", "Alloc (in %)"]
].rename(columns={
    "Dose": "Correct dose",
    "Rec (in %)": "Correct dose rec %",
    "Alloc (in %)": "Correct dose alloc %",
})

print("Recommendation % by dose level")
display(real_rec_pivot)
print("\nAllocation % by dose level")
display(real_alloc_pivot)
print("\nCorrect-dose summary")
display(correct_summary)

Recommendation % by dose level


Dose                        0      1      2      3      4      5
Scenario Algorithm                                              
a = 0.4  3 + 3          85.00  11.67   2.33   1.00   0.00   0.00
         CRM            52.67  25.33  11.00   6.00   3.00   2.00
         UCB            37.33  27.00  20.00  12.00   2.67   1.00
         SEEDA          35.00  34.33  30.67   0.00   0.00   0.00
         SEEDA Plateau  96.00   4.00   0.00   0.00   0.00   0.00
a = 1.0  3 + 3          18.00  27.67  30.67  20.00   3.67   0.00
         CRM            12.00  21.33  32.67  20.67  11.67   1.67
         UCB             8.33  36.67  31.67  21.00   2.33   0.00
         SEEDA           0.67  16.33  56.33  25.33   1.33   0.00
         SEEDA Plateau  87.33  12.33   0.33   0.00   0.00   0.00
a = 1.3  3 + 3           4.33  14.67  44.67  28.00   8.33   0.00
         CRM             6.00  13.00  28.67  38.33  11.33   2.67
         UCB             3.00  17.33  50.33  20.67   8.33   0.33
         SEEDA           0.67   8.33  64.00  23.67   3.33   0.00
         SEEDA Plateau  93.33   6.33   0.33   0.00   0.00   0.00
a = 3.4  3 + 3           0.67   2.33  10.67  23.00  61.00   2.33
         CRM             1.00   3.33  14.00  15.00  28.00  38.67
         UCB             0.67   0.67   8.00  27.67  44.00  19.00
         SEEDA           0.00   2.67  27.33  29.33  35.33   5.33
         SEEDA Plateau  88.67  11.00   0.33   0.00   0.00   0.00


Allocation % by dose level


Dose                        0      1      2      3      4      5
Scenario Algorithm                                              
a = 0.4  3 + 3          82.23  13.97   2.90   0.80   0.10   0.00
         CRM            54.07  22.37  10.33   6.43   4.00   2.80
         UCB            24.17  20.57  17.73  15.17  11.60  10.77
         SEEDA          32.87  41.40  21.20   4.53   0.00   0.00
         SEEDA Plateau  33.17  44.90  19.73   2.20   0.00   0.00
a = 1.0  3 + 3          24.63  29.13  26.37  15.47   4.00   0.40
         CRM            20.80  25.70  27.83  15.10   8.10   2.47
         UCB            12.27  23.00  22.13  20.13  12.00  10.47
         SEEDA          12.03  19.60  35.43  30.80   2.13   0.00
         SEEDA Plateau  14.27  22.47  36.90  25.57   0.80   0.00
a = 1.3  3 + 3          13.93  20.27  35.03  22.47   7.30   1.00
         CRM            15.40  19.80  26.80  27.37   7.83   2.80
         UCB            10.83  15.93  27.43  19.20  15.73  10.87
         SEEDA          12.33  17.83  34.70  27.57   7.57   0.00
         SEEDA Plateau  12.60  23.23  40.83  21.97   1.37   0.00
a = 3.4  3 + 3          10.60  11.83  16.67  21.23  27.50  12.17
         CRM            10.90  12.57  19.37  17.03  19.70  20.43
         UCB            10.20  10.27  12.87  19.63  26.87  20.17
         SEEDA          10.47  12.73  20.57  21.53  22.97  11.73
         SEEDA Plateau  13.27  21.27  33.50  23.90   7.53   0.53


Correct-dose summary


Correct dose  Is MTD  Correct dose rec %  \
Scenario Algorithm                                                 
a = 0.4  3 + 3                     0    True               85.00   
         CRM                       0    True               52.67   
         UCB                       0    True               37.33   
         SEEDA                     0    True               35.00   
         SEEDA Plateau             0    True               96.00   
a = 1.0  3 + 3                     3    True               20.00   
         CRM                       3    True               20.67   
         UCB                       3    True               21.00   
         SEEDA                     2   False               56.33   
         SEEDA Plateau             2   False                0.33   
a = 1.3  3 + 3                     3    True               28.00   
         CRM                       3    True               38.33   
         UCB                       3    True               20.67   
         SEEDA                     2   False               64.00   
         SEEDA Plateau             2   False                0.33   
a = 3.4  3 + 3                     5    True                2.33   
         CRM                       5    True               38.67   
         UCB                       5    True               19.00   
         SEEDA                     2   False               27.33   
         SEEDA Plateau             2   False                0.33   

                        Correct dose alloc %  
Scenario Algorithm                            
a = 0.4  3 + 3                         82.23  
         CRM                           54.07  
         UCB                           24.17  
         SEEDA                         32.87  
         SEEDA Plateau                 33.17  
a = 1.0  3 + 3                         15.47  
         CRM                           15.10  
         UCB                           20.13  
         SEEDA                         35.43  
         SEEDA Plateau                 36.90  
a = 1.3  3 + 3                         22.47  
         CRM                           27.37  
         UCB                           19.20  
         SEEDA                         34.70  
         SEEDA Plateau                 40.83  
a = 3.4  3 + 3                         12.17  
         CRM                           20.43  
         UCB                           20.17  
         SEEDA                         20.57  
         SEEDA Plateau                 33.50

#### Asymptotic

In [17]:
N_ASYM_COHORTS = 300

Run the simulations and collect results in a number of dictionaries, for later analysis.

In [18]:
# Reinitialise the maps for the asymptotic setting:
a_algo_rec_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
cohort_algo_rec_map = {
    a_key(a): {
        cohort_key(cohort): {
            algo: [] for algo in ALGOS
        } for cohort in range(N_ASYM_COHORTS)
    } for a in DOSE_LEVELS.keys()
}
a_algo_alloc_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}
a_algo_n_dle_map = {
    a_key(a): {
        algo: [] for algo in ALGOS
    } for a in DOSE_LEVELS.keys()
}

# Run the asymptotic simulation:
for a, dose_levels in DOSE_LEVELS.items():
    for _ in range(N_TRIALS):
        # To include more dose escalators, add a similar simulation call here.
        tpt_dose_escalator = ThreePlusThreeDoseEscalator(
            dose_levels=dose_levels
        )
        add_simulations(tpt_dose_escalator, a, ALGOS[0], n_cohorts=N_ASYM_COHORTS)
        
        crm_dose_escalator = CRMDoseEscalator(
            dose_levels=dose_levels, 
            target_toxicity_level=TTL, 
            estimator=AveragingEstimator(), 
            conservative=False
        )
        add_simulations(crm_dose_escalator, a, ALGOS[1], n_cohorts=N_ASYM_COHORTS)
        
        ucb_dose_escalator = UCBDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            estimator=AveragingEstimator(),
            ucb_coefficient=UCB_COEFF,
        )
        add_simulations(ucb_dose_escalator, a, ALGOS[2], n_cohorts=N_ASYM_COHORTS)
        
        seeda_dose_escalator = SEEDADoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            no_skip=True
        )
        add_simulations(seeda_dose_escalator, a, ALGOS[3], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

        seedapl_dose_escalator = SEEDAPlateauDoseEscalator(
            dose_levels=dose_levels,
            target_toxicity_level=TTL,
            dose_toxicity_curve=dose_toxic_curve,
            p_hat=P_HAT,
            q_hat=Q_HAT,
            seed=0,
            eta=ETA,
            no_skip=True
        )
        add_simulations(seedapl_dose_escalator, a, ALGOS[4], N_EFFICATE, n_cohorts=N_ASYM_COHORTS)

Plot the final proposals made (for the last cohort) by each of the algorithm, across the trial runs.

In [19]:
# Plot the dose recommendations:
plot_dose_proposals(
    N_LEVELS, 
    N_TRIALS, 
    a_algo_rec_map, 
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of MTD recommendations",
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_algo_dose_rec_proposals.png",
)

Plot the distribution of dose allocations, pooled across all cohorts and trial runs.

In [ ]:
plot_dose_proposals(
    N_LEVELS,
    N_TRIALS * N_ASYM_COHORTS,
    a_algo_alloc_map,
    CORRECT_DOSES,
    mtds=CORRECT_MTDS,
    title_text="Number of dose allocations",
    img_path=f"plots/{timestamp}/Asymptotic/Allocations/asym_algo_dose_allocations.png",
)

Plot the proposal accuracy (whether it matches the MTD) for each cohort so that we can see the time evolution of our `DoseEscalator`'s proposals. 

This is also across all the trial runs, and one plot for each dose-toxicity curve.

In [21]:
plot_acc_progression(
    N_ASYM_COHORTS, 
    cohort_algo_rec_map,
    CORRECT_DOSES,
    show_fig=False, 
    img_path=f"plots/{timestamp}/Asymptotic/Recommendations/asym_dose_rec_proposal_acc_progression.png",
)

Make a table with the metrics.

In [22]:
asym_table = build_results_table(
    a_algo_rec_map, a_algo_alloc_map, CORRECT_MTDS, N_LEVELS, ALGOS,
    opt_doses=OPTIMAL_DOSES, efficacy_algos=EFFICACY_ALGOS,
)

In [23]:
asym_table

,Scenario,Algorithm,Dose,Is correct,Rec (in %),Alloc (in %)
0,a = 0.4,3 + 3,0,True,83.00,82.92
1,a = 0.4,3 + 3,1,False,14.67,14.71
2,a = 0.4,3 + 3,2,False,2.00,2.04
3,a = 0.4,3 + 3,3,False,0.33,0.34
4,a = 0.4,3 + 3,4,False,0.00,0.00
...,...,...,...,...,...,...
115,a = 3.4,SEEDA Plateau,1,False,6.00,3.05
116,a = 3.4,SEEDA Plateau,2,True,0.00,28.66
117,a = 3.4,SEEDA Plateau,3,False,0.00,30.19
118,a = 3.4,SEEDA Plateau,4,False,0.00,23.33


In [24]:
# Paper-style wide layout (one column per dose level):
asym_rec_pivot = asym_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Rec (in %)", sort=False
)
asym_alloc_pivot = asym_table.pivot_table(
    index=["Scenario", "Algorithm"], columns="Dose", values="Alloc (in %)", sort=False
)

# Correct-dose summary: for each scenario/algorithm, the % recommending and
# allocating that algorithm's correct dose (efficacy-optimal for SEEDA/Plateau,
# toxicity MTD otherwise), plus whether that correct dose is the toxicity MTD.
correct_only = asym_table[asym_table["Is correct"]].copy()
correct_only["Is MTD"] = [
    dose == CORRECT_MTDS[scenario]
    for scenario, dose in zip(correct_only["Scenario"], correct_only["Dose"])
]
correct_summary = correct_only.set_index(["Scenario", "Algorithm"])[
    ["Dose", "Is MTD", "Rec (in %)", "Alloc (in %)"]
].rename(columns={
    "Dose": "Correct dose",
    "Rec (in %)": "Correct dose rec %",
    "Alloc (in %)": "Correct dose alloc %",
})

print("Recommendation % by dose level")
display(asym_rec_pivot)
print("\nAllocation % by dose level")
display(asym_alloc_pivot)
print("\nCorrect-dose summary")
display(correct_summary)

Recommendation % by dose level


Dose                         0      1      2      3      4      5
Scenario Algorithm                                               
a = 0.4  3 + 3           83.00  14.67   2.00   0.33   0.00   0.00
         CRM             48.33  36.67  11.33   3.67   0.00   0.00
         UCB             71.00  27.67   1.33   0.00   0.00   0.00
         SEEDA           66.33   5.33  28.33   0.00   0.00   0.00
         SEEDA Plateau  100.00   0.00   0.00   0.00   0.00   0.00
a = 1.0  3 + 3           12.00  31.67  31.67  21.00   3.67   0.00
         CRM             14.33  19.00  26.33  32.67   7.67   0.00
         UCB             11.67  38.00  39.67  10.67   0.00   0.00
         SEEDA            0.00   0.00  60.00  39.33   0.67   0.00
         SEEDA Plateau   96.00   4.00   0.00   0.00   0.00   0.00
a = 1.3  3 + 3            4.67  17.67  41.33  28.67   7.67   0.00
         CRM              8.33  13.00  28.00  33.33  16.67   0.67
         UCB              3.67  24.33  64.67   7.33   0.00   0.00
         SEEDA            0.00   0.00  61.00  37.00   2.00   0.00
         SEEDA Plateau   95.00   5.00   0.00   0.00   0.00   0.00
a = 3.4  3 + 3            1.00   0.67   9.33  24.33  64.67   0.00
         CRM              2.33   8.00  10.67  14.67  31.33  33.00
         UCB              0.00   0.67   9.00  34.67  49.33   6.33
         SEEDA            0.00   0.00  25.00  26.67  29.33  19.00
         SEEDA Plateau   94.00   6.00   0.00   0.00   0.00   0.00


Allocation % by dose level


Dose                        0      1      2      3      4      5
Scenario Algorithm                                              
a = 0.4  3 + 3          82.92  14.71   2.04   0.34   0.00   0.00
         CRM            48.27  36.40  11.04   4.04   0.15   0.10
         UCB            65.72  27.53   4.09   1.44   0.75   0.47
         SEEDA          93.17   5.70   0.99   0.14   0.00   0.00
         SEEDA Plateau  93.82   5.32   0.79   0.06   0.00   0.00
a = 1.0  3 + 3          12.27  31.67  31.53  20.83   3.68   0.02
         CRM            14.61  19.15  26.21  32.28   7.63   0.12
         UCB            11.52  36.81  38.32  12.49   0.49   0.37
         SEEDA           2.15   5.40  48.14  44.21   0.11   0.00
         SEEDA Plateau   0.51   4.99  49.48  44.95   0.07   0.00
a = 1.3  3 + 3           4.99  17.82  41.07  28.45   7.63   0.03
         CRM             8.64  13.21  27.75  33.03  16.29   1.07
         UCB             4.01  23.75  61.35   9.77   0.73   0.39
         SEEDA           2.12   5.54  48.75  42.61   0.98   0.00
         SEEDA Plateau   0.53   4.94  49.69  44.40   0.44   0.00
a = 3.4  3 + 3           1.33   1.01   9.55  24.24  63.39   0.48
         CRM             2.66   8.27  10.86  14.73  31.01  32.48
         UCB             0.35   1.10   9.19  33.78  47.66   7.92
         SEEDA           1.75   4.21  23.52  23.62  24.41  22.49
         SEEDA Plateau   0.47   3.05  28.66  30.19  23.33  14.30


Correct-dose summary


Correct dose  Is MTD  Correct dose rec %  \
Scenario Algorithm                                                 
a = 0.4  3 + 3                     0    True               83.00   
         CRM                       0    True               48.33   
         UCB                       0    True               71.00   
         SEEDA                     0    True               66.33   
         SEEDA Plateau             0    True              100.00   
a = 1.0  3 + 3                     3    True               21.00   
         CRM                       3    True               32.67   
         UCB                       3    True               10.67   
         SEEDA                     2   False               60.00   
         SEEDA Plateau             2   False                0.00   
a = 1.3  3 + 3                     3    True               28.67   
         CRM                       3    True               33.33   
         UCB                       3    True                7.33   
         SEEDA                     2   False               61.00   
         SEEDA Plateau             2   False                0.00   
a = 3.4  3 + 3                     5    True                0.00   
         CRM                       5    True               33.00   
         UCB                       5    True                6.33   
         SEEDA                     2   False               25.00   
         SEEDA Plateau             2   False                0.00   

                        Correct dose alloc %  
Scenario Algorithm                            
a = 0.4  3 + 3                         82.92  
         CRM                           48.27  
         UCB                           65.72  
         SEEDA                         93.17  
         SEEDA Plateau                 93.82  
a = 1.0  3 + 3                         20.83  
         CRM                           32.28  
         UCB                           12.49  
         SEEDA                         48.14  
         SEEDA Plateau                 49.48  
a = 1.3  3 + 3                         28.45  
         CRM                           33.03  
         UCB                            9.77  
         SEEDA                         48.75  
         SEEDA Plateau                 49.69  
a = 3.4  3 + 3                          0.48  
         CRM                           32.48  
         UCB                            7.92  
         SEEDA                         23.52  
         SEEDA Plateau                 28.66